**what a prompt is made of** — the building blocks you can combine when writing one.

## Simple explanation

A prompt = the message you send to an AI to tell it what you want.

It can include up to 6 parts:

| Part | What it does | Example from the text |
|---|---|---|
| **Instruction** | The main task to do | "Summarize the following customer feedback" |
| **Question** | A specific thing to answer | "What are the main issues reported by customers?" |
| **Context** | Background info the model needs | "Feedback is from users of a food delivery app" |
| **Examples** | Sample input → output, so the model learns the pattern | `"Delivery was late..." → "Delivery Issue"` |
| **Constraints** | Rules/limits it must follow | "Use only the feedback", "Under 100 words" |
| **Output Format** | How the answer should be structured | "Return as: 1. Issue 2. Frequency 3. Explanation" |


# Zero-Shot Prompting

The model is given a task without any examples.

Example:

Classify the sentiment:
"The service was excellent."

# One-Shot Prompting

The model is given one example before the actual task.

Example:

Example:
"The product is amazing." → Positive

Now classify:
"The service was terrible."

# Few-Shot Prompting

The model is given multiple examples before the actual task.

Example:

"The product is amazing." → Positive
"The service was terrible." → Negative
"The food was okay." → Neutral

Now classify:
"The delivery was very fast."

## Simple difference:

Zero-shot → 0 examples
One-shot  → 1 example
Few-shot  → Multiple examples

In [ ]:
!uv pip install -q langchain_groq

# PromptTemplate
- single flat text prompt
- Creates a single text prompt

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_groq import ChatGroq

In [ ]:
from google.colab import userdata
api_key = userdata.get('grok_api')

In [ ]:
model = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.0,
    max_retries=2,
    api_key = api_key
)

In [ ]:
prompt = PromptTemplate.from_template(
    """
You are an AI instructor.

Explain {topic} to {audience}.

Requirements:
- Use simple English
- Include one practical example
- Keep the answer under {word_limit} words
"""
)

In [ ]:
chain = prompt | model

In [ ]:
response = chain.invoke(
    {
        "topic": "Vector Database",
        "audience": "beginner developers",
        "word_limit": 200
    }
)

In [ ]:
print(response.content)

# Jinja2 templating with PromptTemplate (variables, if/elif/else)

-  Jinja lets ONE template adapt to different situations using
  variables, {% if %}/{% elif %}/{% else %}, instead of writing a
  separate hardcoded prompt string for every case.

In [ ]:
from langchain_core.prompts import PromptTemplate

template = """
You are an AI instructor.

Explain {{ topic }} to {{ audience }}.

{% if include_example %}
Include one practical example.
{% endif %}

{% if level == "beginner" %}
Use very simple English and avoid complex terminology.
{% elif level == "advanced" %}
Include technical details and architecture.
{% else %}
Use moderate technical depth.
{% endif %}
"""

"topic": "RAG",
"audience": "Python developers",
"include_example": True,
"level": "beginner"

You are an AI instructor.
Explain RAG to Python developers.
Include one practical example.
Use very simple English and avoid complex terminology.

In [ ]:
prompt = PromptTemplate.from_template(
    template,
    template_format="jinja2"
)

In [ ]:
prompt

In [ ]:
formatted_prompt = prompt.invoke(
    {
        "topic": "RAG",
        "audience": "Python developers",
        "include_example": True,
        "level": "beginner"
    }
)

In [31]:
print(formatted_prompt.to_string())


You are an AI instructor.

Explain RAG to Python developers.


Include one practical example.



Use very simple English and avoid complex terminology.



# ChatPromptTemplate

- role-based messages

-  Creates structured chat messages with roles
   such as system, human, and assistant

-  structured messages with roles (system/human/assistant)
 Real chat-model APIs expect role-based messages, so ChatPromptTemplate
 is what production systems actually send.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

In [ ]:
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are an experienced AI instructor. Use simple English."
        ),
        (
            "human",
            "Explain {topic} to {audience}. Include one example."
        )
    ]
)

In [ ]:
chain = prompt | model


In [ ]:
response = chain.invoke(
    {
        "topic": "Vector Database",
        "audience": "beginner developers"
    }
)

In [ ]:
print(response.content)

In [ ]:
print(formatted_prompt.to_string())

# Jinja2 with ChatPromptTemplate (roles + variables + if + for loop)

### using if conditon

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

In [32]:
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
You are an AI instructor.

{% if level == "beginner" %}
Use very simple English and avoid complex terminology.
{% elif level == "advanced" %}
Include technical details and architecture.
{% else %}
Use moderate technical depth.
{% endif %}
"""
        ),
        (
            "human",
            """
Explain {{ topic }} to {{ audience }}.

{% if include_example %}
Include one practical example.
{% endif %}
"""
        )
    ],
    template_format="jinja2"
)


In [33]:

formatted_prompt = prompt.invoke(
    {
        "topic": "RAG",
        "audience": "Python developers",
        "include_example": True,
        "level": "beginner"
    }
)

In [34]:
user_question="explain me about the RAG?"

In [35]:
for message in formatted_prompt.to_messages():
    print(message.type.upper())
    print(message.content)

SYSTEM

You are an AI instructor.


Use very simple English and avoid complex terminology.

HUMAN

Explain RAG to Python developers.


Include one practical example.



### using loop

In [36]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
You are an enterprise AI assistant.

Follow these rules:

{% for rule in rules %}
- {{ rule }}
{% endfor %}
"""
        ),
        (
            "human",
            """
Question:
{{ question }}

{% if output_format == "json" %}
Return the response in JSON format.
{% else %}
Return the response in Markdown.
{% endif %}
"""
        )
    ],
    template_format="jinja2"
)

In [37]:
result = prompt.invoke(
    {
        "rules": [
            "Do not fabricate information",
            "Keep the answer concise",
            "Use only the supplied context"
        ],
        "question": "What is RAG?",
        "output_format": "json"
    }
)

In [38]:
for message in result.to_messages():
    print(message.type, ":", message.content)

system : 
You are an enterprise AI assistant.

Follow these rules:


- Do not fabricate information

- Keep the answer concise

- Use only the supplied context

human : 
Question:
What is RAG?


Return the response in JSON format.



# Loading prompts.json (flat strings) + plain .format()

- prompts.json stores plain {placeholder} strings, no roles, no  versioning. Works for quick scripts but is NOT how a real chat-model app is structured — no system/human separation, no way to track  which version of the prompt produced a given output.

In [39]:
import json

with open("prompts.json", "r", encoding="utf-8") as file:
    prompts = json.load(file)

In [40]:
prompts

{'rag_prompt': 'Answer the question using only the provided context.\n\nContext: {context}\n\nQuestion: {question}',
 'summary_prompt': 'Summarize the following text in 3 bullet points.\n\nText: {text}',
 'classification_prompt': 'Classify the following text as Positive, Negative, or Neutral.\n\nText: {text}',
 'code_review_prompt': 'Review the following Python code and identify any bugs.\n\nCode: {code}'}

In [41]:
prompt = prompts["rag_prompt"]

In [42]:
prompt

'Answer the question using only the provided context.\n\nContext: {context}\n\nQuestion: {question}'

In [43]:
final_prompt = prompt.format(
    context="Employees receive 24 paid leaves every year.",
    question="How many paid leaves do employees receive?"
)


In [44]:
final_prompt

'Answer the question using only the provided context.\n\nContext: Employees receive 24 paid leaves every year.\n\nQuestion: How many paid leaves do employees receive?'

In [45]:
print(final_prompt)

Answer the question using only the provided context.

Context: Employees receive 24 paid leaves every year.

Question: How many paid leaves do employees receive?


# Loading prompt.json (versioned, role-based, Jinja2) via load_prompt()

 prompt.json is the production-style file: each prompt has a "version" field, role-based "messages", and Jinja2 formatting. load_prompt() centralizes ALL prompt loading behind one function — no part of the app should build prompts by hand once this exists.

In [46]:
import json
from langchain_core.prompts import ChatPromptTemplate

In [47]:
def load_prompt(prompt_name):

    with open("prompt.json", "r", encoding="utf-8") as file:
        prompts = json.load(file)

    if prompt_name not in prompts:
        raise ValueError(
            f"Prompt '{prompt_name}' not found."
        )

    config = prompts[prompt_name]

    messages = [
        (message["role"], message["template"])
        for message in config["messages"]
    ]

    prompt = ChatPromptTemplate.from_messages(
        messages,
        template_format=config["template_format"]
    )

    return prompt

In [48]:
prompt = load_prompt("rag_prompt")

In [49]:
prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template="You are a RAG assistant. Answer only from the provided context. If the answer is not available, say 'I do not have enough information.'", template_format='jinja2'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='Context:\n{{ context }}\n\nQuestion:\n{{ question }}', template_format='jinja2'), additional_kwargs={})])

In [50]:
result = prompt.invoke(
    {
        "context": """
        Employees receive 24 paid leaves every year.
        Maximum 10 unused leaves can be carried forward.
        """,

        "question": "How many paid leaves are available?"
    }
)

In [51]:
for message in result.to_messages():
    print(message.type.upper())
    print(message.content)

SYSTEM
You are a RAG assistant. Answer only from the provided context. If the answer is not available, say 'I do not have enough information.'
HUMAN
Context:

        Employees receive 24 paid leaves every year.
        Maximum 10 unused leaves can be carried forward.
        

Question:
How many paid leaves are available?


In [52]:
prompt = load_prompt("summarization_prompt")

In [53]:
prompt

ChatPromptTemplate(input_variables=['num_points', 'text'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a professional text summarizer.', template_format='jinja2'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['num_points', 'text'], input_types={}, partial_variables={}, template='Summarize the following text in {{ num_points }} bullet points:\n\n{{ text }}', template_format='jinja2'), additional_kwargs={})])

In [54]:
result = prompt.invoke(
    {
        "num_points": 3,
        "text": "How many paid leaves are available?"
    }
)

In [55]:
for message in result.to_messages():
    print(message.type.upper())
    print(message.content)

SYSTEM
You are a professional text summarizer.
HUMAN
Summarize the following text in 3 bullet points:

How many paid leaves are available?


In [56]:
from langchain_groq import ChatGroq
model = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.0,
    max_retries=2,
    api_key = api_key
)

prompt = load_prompt(
    "rag_prompt"
)

chain = prompt | model

response = chain.invoke(
    {
        "context": """
        Employees receive 24 paid leaves annually.
        """,

        "question":
        "How many paid leaves do employees receive?"
    }
)

print(response.content)

Employees receive 24 paid leaves annually.


prompts.json
     │
     ├── rag_prompt
     ├── summarization_prompt
     ├── classification_prompt
     └── code_review_prompt
              ↓
        load_prompt()
              ↓
     ChatPromptTemplate
              ↓
             LLM